# PhysIR — Pilot A: Baseline (no physics losses) (Kaggle)

**Purpose:** 50-epoch baseline run: L1 + Edge + Enhance only. Establishes the PSNR/SSIM/LPIPS reference ceiling that Pilots B/C/D are measured against.

**Watch:** Sanity check: epoch-1 loss should be 0.05–0.15. PSNR at epoch 50 is the baseline number to beat. Perceptual scores at ep 10/30/50 are the baseline for all three no-ref metrics.

**Accelerator:** Set *Settings → Accelerator → GPU T4 x2* (or P100).

## 1. Install dependencies

In [ ]:
!pip install -q lpips pytorch-msssim einops ptflops pyiqa


## 2. Paths — edit before running

In [ ]:
import os, sys

# ── EDIT THESE ────────────────────────────────────────────────────────────────
CODE_DIR     = '/kaggle/working/PhysIR'         # path to repo code
LOLBLUR_ROOT = '/kaggle/input/lolblur/LOLBlur'  # your dataset slug

# Persistent checkpoint dir.
# /kaggle/working is saved as output between sessions (re-attach via Add Data).
# Swap to a Drive path if running in Colab: '/content/drive/MyDrive/PhysIR/ckpts'
CHECKPOINT_DIR = '/kaggle/working'
# ──────────────────────────────────────────────────────────────────────────────

WORKING_DIR = '/kaggle/working'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Code:        {CODE_DIR}')
print(f'Dataset:     {LOLBLUR_ROOT}')
print(f'Checkpoints: {CHECKPOINT_DIR}')


## 3. Clone PhysIR repo *(skip if mounted as Kaggle dataset)*

In [ ]:
if not os.path.isdir(CODE_DIR):
    GITHUB_URL = 'https://github.com/<your-username>/PhysIR.git'  # EDIT
    !git clone {GITHUB_URL} {CODE_DIR}
else:
    print('Code directory already present.')

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
os.chdir(CODE_DIR)
print('cwd:', os.getcwd())


## 4. Train/val split

In [ ]:
import random
from pathlib import Path

SUBS       = ['low_blur_noise', 'high_sharp_scaled']
TRAIN_SRC  = Path(LOLBLUR_ROOT) / 'train'
SPLIT_ROOT = Path(WORKING_DIR) / 'LOLBlur_split'

random.seed(42)
clips = sorted([d.name for d in (TRAIN_SRC / 'low_blur_noise').iterdir() if d.is_dir()])
n_val = max(1, int(len(clips) * 0.1))
val_clips   = set(random.sample(clips, n_val))
train_clips = [c for c in clips if c not in val_clips]
print(f'Clips — train: {len(train_clips)}, val: {len(val_clips)}')

for split_name, clip_list in [('physir_train', train_clips), ('physir_val', val_clips)]:
    for sub in SUBS:
        dst_base = SPLIT_ROOT / split_name / sub
        dst_base.mkdir(parents=True, exist_ok=True)
        for clip in clip_list:
            src_path = (TRAIN_SRC / sub / clip).resolve()
            dst_path = dst_base / clip
            if not dst_path.exists():
                dst_path.symlink_to(src_path)

PHYSIR_TRAIN = str(SPLIT_ROOT / 'physir_train')
PHYSIR_VAL   = str(SPLIT_ROOT / 'physir_val')
print(f'  train: {PHYSIR_TRAIN}')
print(f'  val:   {PHYSIR_VAL}')


## 5. Config — Pilot A: Baseline (no physics losses)

In [ ]:
import yaml

cfg = {
    'datasets': {
        'name': 'LOLBlur',
        'train': {
            'low_path':   f'{PHYSIR_TRAIN}/low_blur_noise',
            'high_path':  f'{PHYSIR_TRAIN}/high_sharp_scaled',
            'crop_size':  256, 'batch_size': 4, 'n_workers': 4, 'use_flips': True,
        },
        'val': {
            'test_path': PHYSIR_VAL, 'batch_size_test': 1,
            'verbose': False, 'n_workers': 4,
        },
    },
    'network': {
        'name': 'DarkIR', 'img_channels': 3, 'width': 32,
        'middle_blk_num_enc': 2, 'middle_blk_num_dec': 2,
        'enc_blk_nums': [1, 2, 3], 'dec_blk_nums': [3, 1, 1],
        'dilations': [1, 4, 9], 'extra_depth_wise': True,
    },
    'save': {
        'pretrained': None,
        'path': f'{CHECKPOINT_DIR}/pilot_A_baseline_latest.pt',
        'best': f'{CHECKPOINT_DIR}/pilot_A_baseline_best.pt',
        'save_freq': 5, 'eval_freq': 5,
        'perceptual_eval_epochs': [10, 30, 50],
    },
    'pixel_criterion': 'l1', 'perceptual': False,
    'edge': True, 'edge_weight': 0.05, 'edge_criterion': 'l2', 'edge_reduction': 'mean',
    'frequency': False,
    'enhance': True, 'enhance_weight': 0.01,
    'enhance_criterion': 'l1', 'enhance_reduction': 'mean',
    'physics_losses': {'enabled': False},
    'train': {
        'epochs': 50, 'lr_initial': 0.00028, 'weight_decay': 0.001,
        'lr_scheme': 'CosineAnnealing', 'eta_min': 1e-6, 'grad_clip': 1.0,
    },
    'wandb': {'init': False, 'name': 'pilot_A_baseline'},
    'Resize': False,
}

cfg_path = f'{WORKING_DIR}/pilot_A_baseline_kaggle.yml'
with open(cfg_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)
print(f'Config: {cfg_path}')


## 6. Resume detection

In [ ]:
# train.py auto-resumes when save.path exists on disk.
# This cell checks for an existing checkpoint so you know where training will pick up.

latest_ckpt = f'{CHECKPOINT_DIR}/pilot_A_baseline_latest.pt'

if os.path.isfile(latest_ckpt):
    size_mb = os.path.getsize(latest_ckpt) / 1e6
    import torch as _t
    _ckpt = _t.load(latest_ckpt, map_location='cpu', weights_only=False)
    resume_ep = _ckpt.get('epoch', '?')
    del _ckpt
    print(f'✅ Checkpoint found ({size_mb:.1f} MB) — resuming from epoch {resume_ep}')
else:
    print('No checkpoint found — training from scratch.')


## 7. Train

In [ ]:
!torchrun --nproc_per_node=2 {CODE_DIR}/train.py -p {cfg_path}


## 8. Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

metrics_dir = Path(CODE_DIR) / '.metrics' / 'pilot_A_baseline'
log_path  = metrics_dir / 'training_log.csv'
perc_path = metrics_dir / 'perceptual_log.csv'

if not log_path.exists():
    print('Log not found — training may still be running or path is wrong.')
else:
    df = pd.read_csv(log_path)
    # Empty strings written by MetricsLogger are not NaN — replace them
    df.replace('', float('nan'), inplace=True)
    for col in ['val_psnr', 'val_ssim', 'val_lpips']:
        if col not in df.columns:
            df[col] = float('nan')
    val = df.dropna(subset=['val_psnr'])

    fig, axes = plt.subplots(1, 5, figsize=(25, 4))
    ax = iter(axes)
    df.plot(x='epoch', y='loss_total', ax=next(ax), title='Total loss')
    val.plot(x='epoch', y='val_psnr',  ax=next(ax), title='Val PSNR (dB)', marker='o')
    val.plot(x='epoch', y='val_ssim',  ax=next(ax), title='Val SSIM',      marker='o')
    val.plot(x='epoch', y='val_lpips', ax=next(ax), title='Val LPIPS ↓',   marker='o')

    if perc_path.exists():
        perc = pd.read_csv(perc_path)
        ax_perc = next(ax)
        perc.plot(x='epoch', y=['nrqm', 'musiq'], ax=ax_perc,
                  title='NRQM↑ / MUSIQ↑  |  NIQE↓ (red)', marker='o')
        ax_perc.twinx().plot(perc['epoch'], perc['niqe'],
                             color='red', linestyle='--', marker='s', label='NIQE↓')
    else:
        next(ax).set_title('Perceptual: pending (ep 10/30/50)')

    plt.tight_layout()
    plt.savefig(f'{WORKING_DIR}/pilot_A_baseline_curves.png', dpi=120)
    plt.show()

    print('\n=== Summary ===')
    if len(val):
        print(f'  Best PSNR  : {val["val_psnr"].max():.4f} dB')
        if val["val_ssim"].notna().any():
            print(f'  Best SSIM  : {val["val_ssim"].max():.4f}')
        if val["val_lpips"].notna().any():
            print(f'  Best LPIPS : {val["val_lpips"].min():.4f}  (lower=better)')
    if perc_path.exists():
        last = perc.iloc[-1]
        ep   = int(last['epoch'])
        print(f'  NIQE  ep{ep}: {last["niqe"]:.4f}  (lower=better)')
        print(f'  NRQM  ep{ep}: {last["nrqm"]:.4f}  (higher=better)')
        print(f'  MUSIQ ep{ep}: {last["musiq"]:.4f}  (higher=better)')